In [ ]:
# -*- coding: utf-8 -*-
import os
import pandas as pd

# =========================
# 1) 路径配置
# =========================
IN_CSV = r"D:\PythonProject\LASSO_FINAL\graph\graph1\results\predictions_Graph1_test_detail.csv"
OUT_CSV = r"D:\PythonProject\LASSO_FINAL\Result\predictions_top20_each_day.csv"

os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

# =========================
# 2) 读取数据
# =========================
df = pd.read_csv(IN_CSV)

need_cols = ["date", "stock", "y_pred", "y_true"]
miss = [c for c in need_cols if c not in df.columns]
if miss:
    raise ValueError(f"缺少列: {miss}")

# =========================
# 3) 基础清洗
# =========================
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["stock"] = df["stock"].astype(str)
df["y_pred"] = pd.to_numeric(df["y_pred"], errors="coerce")
df["y_true"] = pd.to_numeric(df["y_true"], errors="coerce")

df = df.dropna(subset=["date", "stock", "y_pred", "y_true"]).copy()

# =========================
# 4) 每天按 y_pred 排序，保留前20
# =========================
df = df.sort_values(
    by=["date", "y_pred", "stock"],
    ascending=[True, False, True]
).copy()

# 生成每天的排序名次
df["rank"] = df.groupby("date")["y_pred"].rank(method="first", ascending=False).astype(int)

# 保留每天前20
df_top20 = df[df["rank"] <= 20].copy()

# 再按 date + rank 排序，方便查看
df_top20 = df_top20.sort_values(["date", "rank"]).reset_index(drop=True)

# =========================
# 5) 保存结果
# =========================
df_top20.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

# =========================
# 6) 打印信息
# =========================
print(f"原始记录数: {len(df)}")
print(f"保留前20后记录数: {len(df_top20)}")
print(f"交易日数量: {df_top20['date'].nunique()}")
print(f"结果已保存到: {OUT_CSV}")

print("\n前几行示例:")
print(df_top20.head(30))

ST股票挑选

In [3]:
#ST股票挑选
import pandas as pd

# =========================
# 1) 文件路径
# =========================
input_file = r"D:\PythonProject\LASSO_FINAL\data\实验使用主板股票信息.xlsx"
output_file = r"D:\PythonProject\LASSO_FINAL\data\ST股票名单.xlsx"

# =========================
# 2) 读取 Excel
# =========================
df = pd.read_excel(input_file, sheet_name="Sheet1")

# =========================
# 3) 筛选 ST / *ST 股票
# name 列中只要包含 "ST" 就保留
# na=False 可以避免空值报错
# =========================
st_df = df[df["name"].astype(str).str.contains("ST", na=False)].copy()

# =========================
# 4) 保存结果
# =========================
st_df.to_excel(output_file, index=False)

# =========================
# 5) 打印结果
# =========================
print(f"共筛选出 {len(st_df)} 只 ST 股票")
print(st_df[["ts_code", "name"]].head(20))
print(f"结果已保存到: {output_file}")

共筛选出 127 只 ST 股票
       ts_code   name
2    000004.SZ  *ST国华
86   000430.SZ  *ST张股
87   000488.SZ   ST晨鸣
91   000504.SZ  *ST生物
101  000518.SZ  *ST四环
157  000595.SZ  *ST宝实
167  000608.SZ  *ST阳光
168  000609.SZ   ST中迪
171  000615.SZ  *ST美谷
187  000638.SZ  *ST万方
193  000656.SZ  *ST金科
199  000668.SZ  *ST荣控
200  000669.SZ   ST金鸿
215  000691.SZ  *ST亚太
218  000697.SZ   ST炼石
219  000698.SZ   ST沈化
229  000711.SZ   ST京蓝
249  000736.SZ  *ST中地
255  000752.SZ   ST西发
280  000793.SZ   ST华闻
结果已保存到: D:\PythonProject\LASSO_FINAL\data\ST股票名单.xlsx


#检查panel中是否还存在ST股票


In [4]:
# -*- coding: utf-8 -*-
import os
import pickle
import pandas as pd

# =========================================================
# 1) 路径
# =========================================================
ST_LIST_PATH = r"D:\PythonProject\LASSO_FINAL\data\ST股票名单.xlsx"
PANEL_PATH   = r"D:\PythonProject\LASSO_FINAL\data\panel_expand_selected_cache.pkl"   # 改成你的 panel 文件
OUT_PATH     = r"D:\PythonProject\LASSO_FINAL\data\panel_expand_selected_cache_noST.pkl"

# =========================================================
# 2) 工具函数
# =========================================================
def normalize_code(x):
    """统一股票代码格式为6位数字，例如 000001.SZ -> 000001"""
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if "." in s:
        s = s.split(".")[0]
    s = "".join(ch for ch in s if ch.isdigit())
    if s == "":
        return None
    return s.zfill(6)

def load_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pkl":
        with open(path, "rb") as f:
            return pickle.load(f)
    elif ext == ".csv":
        return pd.read_csv(path, low_memory=False)
    elif ext in [".xls", ".xlsx"]:
        return pd.read_excel(path)
    elif ext == ".parquet":
        return pd.read_parquet(path)
    else:
        raise ValueError(f"不支持的文件格式: {ext}")

def extract_df(obj):
    """尽量从 pkl / dict 中提取 DataFrame"""
    if isinstance(obj, pd.DataFrame):
        return obj
    if isinstance(obj, dict):
        for k in ["panel", "df", "data", "X"]:
            if k in obj and isinstance(obj[k], pd.DataFrame):
                print(f"[INFO] 从字典键 {k} 提取 DataFrame")
                return obj[k]
        for k, v in obj.items():
            if isinstance(v, pd.DataFrame):
                print(f"[INFO] 从字典键 {k} 提取 DataFrame")
                return v
    raise ValueError("读取到的 panel 不是 DataFrame，也没在 dict 中找到 DataFrame。")

def detect_stock_col(df):
    for c in ["ts_code", "stock", "code", "ticker", "symbol"]:
        if c in df.columns:
            return c
    raise ValueError(f"没找到股票代码列。当前列名：{list(df.columns)}")

# =========================================================
# 3) 读取 ST 股票名单
# =========================================================
st_df = pd.read_excel(ST_LIST_PATH)

if "ts_code" not in st_df.columns:
    raise ValueError("ST股票名单.xlsx 中缺少 ts_code 列。")

st_df["code6"] = st_df["ts_code"].apply(normalize_code)
st_code_set = set(st_df["code6"].dropna().unique())

print("=" * 60)
print(f"ST 名单文件路径: {ST_LIST_PATH}")
print(f"ST 股票总数: {len(st_code_set)}")
print("=" * 60)

# =========================================================
# 4) 读取 panel
# =========================================================
panel_obj = load_any(PANEL_PATH)
panel = extract_df(panel_obj).copy()

stock_col = detect_stock_col(panel)
panel["code6"] = panel[stock_col].apply(normalize_code)

panel_stock_set = set(panel["code6"].dropna().unique())

print(f"[INFO] panel 文件路径: {PANEL_PATH}")
print(f"[INFO] panel 股票列: {stock_col}")
print(f"[INFO] panel 总行数: {len(panel)}")
print(f"[INFO] panel 唯一股票数: {len(panel_stock_set)}")

# =========================================================
# 5) 检查是否还有 ST
# =========================================================
st_left = sorted(panel_stock_set & st_code_set)
non_st_left = sorted(panel_stock_set - st_code_set)

print("-" * 60)
print(f"panel 中仍然存在的 ST 股票数: {len(st_left)}")
print(f"panel 中剩余的非 ST 股票数: {len(non_st_left)}")

if len(st_left) > 0:
    print("-" * 60)
    print("仍残留的 ST 股票代码（前100个）:")
    print(st_left[:100])
else:
    print("-" * 60)
    print("panel 中已经没有 ST 股票。")

# =========================================================
# 6) 如需剔除 ST，并保存新 panel
# =========================================================
panel_no_st = panel[~panel["code6"].isin(st_code_set)].copy()
panel_no_st.drop(columns=["code6"], inplace=True, errors="ignore")

new_stock_num = panel_no_st[stock_col].apply(normalize_code).nunique()

print("-" * 60)
print(f"剔除 ST 后，panel 剩余总行数: {len(panel_no_st)}")
print(f"剔除 ST 后，panel 剩余唯一股票数: {new_stock_num}")

with open(OUT_PATH, "wb") as f:
    pickle.dump(panel_no_st, f)

print(f"[保存完成] 新 panel 已保存到: {OUT_PATH}")
print("=" * 60)

ST 名单文件路径: D:\PythonProject\LASSO_FINAL\data\ST股票名单.xlsx
ST 股票总数: 127
[INFO] panel 文件路径: D:\PythonProject\LASSO_FINAL\data\panel_expand_selected_cache.pkl
[INFO] panel 股票列: ts_code
[INFO] panel 总行数: 1673865
[INFO] panel 唯一股票数: 3132
------------------------------------------------------------
panel 中仍然存在的 ST 股票数: 127
panel 中剩余的非 ST 股票数: 3005
------------------------------------------------------------
仍残留的 ST 股票代码（前100个）:
['000004', '000430', '000488', '000504', '000518', '000595', '000608', '000609', '000615', '000638', '000656', '000668', '000669', '000691', '000697', '000698', '000711', '000736', '000752', '000793', '000820', '000903', '000908', '000909', '000929', '000972', '001270', '002005', '002024', '002047', '002058', '002076', '002122', '002168', '002199', '002200', '002211', '002214', '002231', '002253', '002289', '002305', '002306', '002388', '002424', '002485', '002496', '002528', '002529', '002569', '002581', '002586', '002592', '002620', '002630', '002647', '002650', '002

#剔除9%的股票数据

In [5]:
# -*- coding: utf-8 -*-
import pickle
import pandas as pd

# =========================================================
# 1) 路径
# =========================================================
PANEL_PATH = r"D:\PythonProject\LASSO_FINAL\data\panel_expand_selected_cache_noST.pkl"
LIMIT9_PATH = r"D:\PythonProject\LASSO_FINAL\Result\大于9个点涨停.csv"
OUT_PATH = r"D:\PythonProject\LASSO_FINAL\data\panel_expand_selected_cache_noST_no9pct.pkl"

# =========================================================
# 2) 工具函数
# =========================================================
def norm_stock(x):
    """
    股票代码统一成6位数字
    例如:
    000001.SZ -> 000001
    000001    -> 000001
    1         -> 000001
    1.0       -> 000001
    """
    if pd.isna(x):
        return None

    s = str(x).strip().upper()

    # 先去掉交易所后缀
    if "." in s and s.endswith(("SZ", "SH", "BJ")):
        s = s.split(".")[0]

    # 处理像 1.0 这种
    try:
        if s.replace(".", "", 1).isdigit():
            s = str(int(float(s)))
    except:
        pass

    # 只保留数字
    s = "".join(ch for ch in s if ch.isdigit())

    if s == "":
        return None

    return s.zfill(6)


def norm_date(x):
    """
    日期统一成 YYYYMMDD 字符串
    兼容：
    20220106
    2022-01-06
    2022/01/06
    2022-01-06 00:00:00
    Timestamp(...)
    """
    if pd.isna(x):
        return None

    s = str(x).strip()

    # 保留数字
    digits = "".join(ch for ch in s if ch.isdigit())

    # 常见情况:
    # 20220106
    # 20220106000000
    if len(digits) >= 8:
        return digits[:8]

    return None


def extract_panel_df(panel_obj):
    if isinstance(panel_obj, pd.DataFrame):
        return panel_obj.copy()

    if isinstance(panel_obj, dict):
        for k in ["panel", "df", "data", "X"]:
            if k in panel_obj and isinstance(panel_obj[k], pd.DataFrame):
                print(f"[INFO] 从键 {k!r} 提取 DataFrame")
                return panel_obj[k].copy()

        for k, v in panel_obj.items():
            if isinstance(v, pd.DataFrame):
                print(f"[INFO] 从键 {k!r} 提取 DataFrame")
                return v.copy()

    raise ValueError("pkl 中没有找到 DataFrame，请检查 panel 文件结构。")


def detect_stock_col(df):
    for c in ["ts_code", "stock", "code", "ticker", "symbol"]:
        if c in df.columns:
            return c
    raise ValueError(f"panel 中未找到股票代码列，当前列名: {list(df.columns)}")


def detect_date_col(df):
    for c in ["date", "trade_date", "datetime"]:
        if c in df.columns:
            return c
    raise ValueError(f"panel 中未找到日期列，当前列名: {list(df.columns)}")


# =========================================================
# 3) 读取 panel
# =========================================================
with open(PANEL_PATH, "rb") as f:
    panel_obj = pickle.load(f)

panel = extract_panel_df(panel_obj)

stock_col = detect_stock_col(panel)
date_col = detect_date_col(panel)

panel[date_col] = panel[date_col].apply(norm_date)
panel["_stock"] = panel[stock_col].apply(norm_stock)

# 去掉关键键为空的行
panel = panel.dropna(subset=[date_col, "_stock"]).copy()

print("=" * 70)
print("[1] PANEL")
print(f"panel 总行数: {len(panel)}")
print(f"panel 唯一股票数: {panel['_stock'].nunique()}")
print(f"panel 日期数: {panel[date_col].nunique()}")
print("panel 日期示例:", panel[date_col].head(10).tolist())
print("panel 股票示例:", panel["_stock"].head(10).tolist())
print("=" * 70)

# =========================================================
# 4) 读取 9%涨停表
#    宽表结构:
#    trade_date | 000001.SZ | 000002.SZ | ...
# =========================================================
limit_wide = pd.read_csv(LIMIT9_PATH, low_memory=False)

if "trade_date" not in limit_wide.columns:
    raise ValueError(f"9%涨停表缺少 trade_date 列，当前列名: {limit_wide.columns.tolist()[:20]}")

limit_wide["trade_date"] = limit_wide["trade_date"].apply(norm_date)

limit_long = limit_wide.melt(
    id_vars="trade_date",
    var_name="_stock_raw",
    value_name="limit9_flag_raw"
)

limit_long["_stock"] = limit_long["_stock_raw"].apply(norm_stock)
limit_long["limit9_flag"] = pd.to_numeric(limit_long["limit9_flag_raw"], errors="coerce").fillna(0).astype(int)

# 只保留 flag=1
limit_long = limit_long[limit_long["limit9_flag"] == 1].copy()
limit_long = limit_long.dropna(subset=["trade_date", "_stock"]).copy()

print("=" * 70)
print("[2] LIMIT LONG")
print(f"flag=1 行数: {len(limit_long)}")
print(f"唯一股票数: {limit_long['_stock'].nunique()}")
print(f"日期数: {limit_long['trade_date'].nunique()}")
print("limit 日期示例:", limit_long["trade_date"].head(10).tolist())
print("limit 股票示例:", limit_long["_stock"].head(10).tolist())
print("=" * 70)

# =========================================================
# 5) 建立 buy_date=t-1 -> panel_date=t 的映射
# =========================================================
trade_dates = sorted([d for d in limit_wide["trade_date"].dropna().unique() if isinstance(d, str)])

date_map = pd.DataFrame({
    "buy_date": trade_dates[:-1],
    date_col: trade_dates[1:]
})

# limit 表中的 trade_date 是买入日
limit_long = limit_long.rename(columns={"trade_date": "buy_date"})

limit_for_panel = limit_long.merge(date_map, on="buy_date", how="inner")
limit_for_panel = limit_for_panel[[date_col, "_stock", "limit9_flag"]].drop_duplicates(
    subset=[date_col, "_stock"]
)

print("=" * 70)
print("[3] 对齐后 limit_for_panel")
print(f"行数: {len(limit_for_panel)}")
print(f"唯一股票数: {limit_for_panel['_stock'].nunique()}")
print(f"日期数: {limit_for_panel[date_col].nunique()}")
print(limit_for_panel.head(10).to_string(index=False))
print("=" * 70)

# =========================================================
# 6) merge 到 panel
# =========================================================
panel2 = panel.merge(
    limit_for_panel,
    on=[date_col, "_stock"],
    how="left"
)

panel2["limit9_flag"] = panel2["limit9_flag"].fillna(0).astype(int)

print("=" * 70)
print("[4] merge 结果")
print("limit9_flag 分布：")
print(panel2["limit9_flag"].value_counts(dropna=False))
print("命中样本前10行：")
tmp = panel2.loc[panel2["limit9_flag"] == 1, [date_col, "_stock", "limit9_flag"]].head(10)
print(tmp.to_string(index=False) if len(tmp) > 0 else "没有命中")
print("=" * 70)

# =========================================================
# 7) 过滤
# =========================================================
before_rows = len(panel2)
before_stocks = panel2["_stock"].nunique()

drop_mask = panel2["limit9_flag"] == 1
drop_rows = int(drop_mask.sum())

panel_filtered = panel2.loc[~drop_mask].copy()

after_rows = len(panel_filtered)
after_stocks = panel_filtered["_stock"].nunique()

print("=" * 70)
print("[5] 过滤结果")
print("过滤条件：若买入日(t-1) 在 9%涨停表中为1，则删除 panel 中 date=t 的样本")
print(f"剔除样本数: {drop_rows}")
print(f"过滤前总行数: {before_rows}")
print(f"过滤后总行数: {after_rows}")
print(f"过滤前唯一股票数: {before_stocks}")
print(f"过滤后唯一股票数: {after_stocks}")
print("=" * 70)

# =========================================================
# 8) 保存
# =========================================================
panel_filtered.drop(columns=["_stock"], inplace=True, errors="ignore")

with open(OUT_PATH, "wb") as f:
    pickle.dump(panel_filtered, f)

print(f"[保存完成] {OUT_PATH}")

[1] PANEL
panel 总行数: 1605114
panel 唯一股票数: 3005
panel 日期数: 543
panel 日期示例: ['20220401', '20220401', '20220401', '20220401', '20220401', '20220401', '20220401', '20220401', '20220401', '20220401']
panel 股票示例: ['000001', '000002', '000006', '000007', '000008', '000009', '000010', '000011', '000012', '000014']
[2] LIMIT LONG
flag=1 行数: 47816
唯一股票数: 3047
日期数: 936
limit 日期示例: ['20221129', '20240221', '20221111', '20221129', '20240429', '20240517', '20240830', '20240926', '20240927', '20240930']
limit 股票示例: ['000001', '000001', '000002', '000002', '000002', '000002', '000002', '000002', '000002', '000002']
[3] 对齐后 limit_for_panel
行数: 47745
唯一股票数: 3047
日期数: 935
    date _stock  limit9_flag
20221130 000001            1
20240222 000001            1
20221114 000002            1
20221130 000002            1
20240430 000002            1
20240520 000002            1
20240902 000002            1
20240927 000002            1
20240930 000002            1
20241008 000002            1
[4] merge 结果
limit9

In [2]:
# -*- coding: utf-8 -*-
import os
import pickle
import numpy as np
import pandas as pd
from datetime import datetime

# =========================================================
# 配置区域
# =========================================================
PANEL_PATH = r"D:\PythonProject\LASSO_FINAL\data\panel_raw_selected_noST.pkl"
REF_CSV_PATH = r"D:\PythonProject\LASSO_FINAL\data\14.30相对前一日收盘涨幅.csv"

OUT_PANEL_PATH = r"D:\PythonProject\LASSO_FINAL\data\panel_raw_selected_noST_no9pct.pkl"
REMOVED_DETAIL_PATH = r"D:\PythonProject\LASSO_FINAL\data\panel_removed_by_1430_limit_detail.csv"

THRESHOLD = 0.09
SAVE_REMOVED_DETAIL = True

# =========================================================
# 工具函数
# =========================================================
def smart_to_datetime(series):
    """
    尽量稳健地把日期列转成 datetime。
    兼容：
    - 20250102
    - 2025-01-02
    - 2025/01/02
    - 2025-01-02 00:00:00
    """
    s = series.astype(str).str.strip()

    mask_8 = s.str.fullmatch(r"\d{8}")
    out = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")

    if mask_8.any():
        out.loc[mask_8] = pd.to_datetime(s.loc[mask_8], format="%Y%m%d", errors="coerce")

    if (~mask_8).any():
        out.loc[~mask_8] = pd.to_datetime(s.loc[~mask_8], errors="coerce")

    return out


def normalize_stock_code(series):
    """
    统一股票代码格式：去空格、转大写
    """
    return series.astype(str).str.strip().str.upper()


def extract_panel_df(panel_obj):
    if isinstance(panel_obj, pd.DataFrame):
        return panel_obj.copy()

    if isinstance(panel_obj, dict):
        for k in ["panel", "df", "data", "X"]:
            if k in panel_obj and isinstance(panel_obj[k], pd.DataFrame):
                print(f"[INFO] 从键 {k!r} 提取 DataFrame")
                return panel_obj[k].copy()

        for k, v in panel_obj.items():
            if isinstance(v, pd.DataFrame):
                print(f"[INFO] 从键 {k!r} 提取 DataFrame")
                return v.copy()

    raise ValueError("pkl 中没有找到 DataFrame，请检查 panel 文件结构。")


def detect_stock_col(df):
    for c in ["ts_code", "stock", "code", "ticker", "symbol"]:
        if c in df.columns:
            return c
    raise ValueError(f"panel 中未找到股票代码列，当前列名: {list(df.columns)}")


def detect_date_col(df):
    for c in ["date", "trade_date", "datetime"]:
        if c in df.columns:
            return c
    raise ValueError(f"panel 中未找到日期列，当前列名: {list(df.columns)}")


def load_reference_table(ref_csv_path):
    """
    读取宽表形式的 14:30 相对前一日收盘涨幅表，并转成长表：
    原格式：
        trade_date, 000001.SZ, 000002.SZ, ...
    转为：
        ts_code, ref_date, ref_value
    """
    if not os.path.exists(ref_csv_path):
        raise FileNotFoundError(f"未找到参考表文件: {ref_csv_path}")

    print(f"[INFO] 正在读取参考表: {ref_csv_path}")
    ref_df = pd.read_csv(ref_csv_path, low_memory=False)

    if "trade_date" not in ref_df.columns:
        raise ValueError("参考表中未找到 'trade_date' 列。")

    ref_df["trade_date"] = smart_to_datetime(ref_df["trade_date"])
    if ref_df["trade_date"].isna().any():
        bad_cnt = ref_df["trade_date"].isna().sum()
        raise ValueError(f"参考表的 trade_date 有 {bad_cnt} 行无法解析为日期，请检查。")

    stock_cols = [c for c in ref_df.columns if c != "trade_date"]
    if len(stock_cols) == 0:
        raise ValueError("参考表中除 trade_date 外没有股票列。")

    print(f"[INFO] 参考表共有 {len(stock_cols)} 只股票列，开始转成长表...")
    ref_long = ref_df.melt(
        id_vars="trade_date",
        value_vars=stock_cols,
        var_name="ts_code",
        value_name="ref_value"
    )

    ref_long["ts_code"] = normalize_stock_code(ref_long["ts_code"])
    ref_long["ref_value"] = pd.to_numeric(ref_long["ref_value"], errors="coerce")

    # 只保留有效值，保证“上一个有值交易日”的逻辑
    ref_long = ref_long.dropna(subset=["ref_value"]).copy()

    ref_long = ref_long.rename(columns={"trade_date": "ref_date"})
    ref_long = ref_long.sort_values(["ts_code", "ref_date"]).reset_index(drop=True)

    print(f"[INFO] 长表转换完成，有效记录数: {len(ref_long):,}")
    return ref_long


def load_panel(panel_path):
    """
    读取 panel，并规范化 ts_code / date
    """
    if not os.path.exists(panel_path):
        raise FileNotFoundError(f"未找到 panel 文件: {panel_path}")

    print(f"[INFO] 正在读取 panel: {panel_path}")
    with open(panel_path, "rb") as f:
        panel_obj = pickle.load(f)

    panel_df = extract_panel_df(panel_obj)

    stock_col = detect_stock_col(panel_df)
    date_col = detect_date_col(panel_df)

    panel_df[stock_col] = normalize_stock_code(panel_df[stock_col])
    panel_df[date_col] = smart_to_datetime(panel_df[date_col])

    if panel_df[date_col].isna().any():
        bad_cnt = panel_df[date_col].isna().sum()
        raise ValueError(f"panel 的日期列 {date_col} 有 {bad_cnt} 行无法解析为日期，请检查。")

    return panel_df, stock_col, date_col


def attach_previous_valid_value(panel_df, stock_col, date_col, ref_long):
    """
    给 panel 每一行匹配：
    同一股票、当前日期之前最近一个“有值”的 ref_value

    这里“之前”是严格小于当前 date，
    所以 panel 的 date=t 时，匹配到的是买入日 t-1 的记录。
    """
    print("[INFO] 正在匹配每一行对应的“上一个有效交易日”的 14:30 涨幅...")

    work = panel_df.copy()
    work["__row_id__"] = np.arange(len(work))
    work["ref_date"] = pd.NaT
    work["ref_value"] = np.nan

    ref_dict = {}
    for code, grp in ref_long.groupby("ts_code"):
        g = grp.sort_values("ref_date").reset_index(drop=True).copy()
        ref_dict[code] = g

    matched_count = 0
    no_prev_count = 0
    no_code_count = 0

    for code, idx in work.groupby(stock_col).groups.items():
        idx = list(idx)
        panel_sub = work.loc[idx].copy().sort_values(date_col)

        if code not in ref_dict:
            no_code_count += len(panel_sub)
            continue

        ref_sub = ref_dict[code]
        ref_dates = ref_sub["ref_date"].values
        ref_values = ref_sub["ref_value"].values

        # 找严格小于当前 date 的最后一个位置 => t-1
        pos = np.searchsorted(ref_dates, panel_sub[date_col].values, side="left") - 1

        valid_mask = pos >= 0
        matched_count += valid_mask.sum()
        no_prev_count += (~valid_mask).sum()

        if valid_mask.any():
            valid_idx = panel_sub.index[valid_mask]
            valid_pos = pos[valid_mask]

            work.loc[valid_idx, "ref_date"] = ref_dates[valid_pos]
            work.loc[valid_idx, "ref_value"] = ref_values[valid_pos]

    print(f"[INFO] 匹配完成：")
    print(f"       成功匹配到上一个有效交易日的行数: {matched_count}")
    print(f"       因没有更早有效交易日而未匹配的行数: {no_prev_count}")
    print(f"       因参考表中不存在该股票而未匹配的行数: {no_code_count}")

    work = work.sort_values("__row_id__").reset_index(drop=True)
    return work


def filter_panel(panel_with_ref, stock_col, date_col, out_panel_path,
                 removed_detail_path=None, save_removed_detail=True):
    """
    根据上一个有效交易日的 14:30 涨幅筛选 panel
    删除条件：ref_value >= THRESHOLD
    """
    remove_mask = panel_with_ref["ref_value"] >= THRESHOLD
    remove_mask = remove_mask.fillna(False)

    keep_mask = ~remove_mask

    removed_count = int(remove_mask.sum())
    kept_count = int(keep_mask.sum())

    print("\n================ 筛选统计 ================")
    print(f"原始样本数: {len(panel_with_ref):,}")
    print(f"删除样本数: {removed_count:,}")
    print(f"保留样本数: {kept_count:,}")
    print("=========================================\n")

    removed_detail = panel_with_ref.loc[remove_mask].copy()
    filtered_panel = panel_with_ref.loc[keep_mask].copy()

    helper_cols = ["__row_id__", "ref_date", "ref_value"]
    filtered_panel = filtered_panel.drop(columns=helper_cols, errors="ignore").reset_index(drop=True)

    with open(out_panel_path, "wb") as f:
        pickle.dump(filtered_panel, f)

    print(f"[INFO] 筛选后的 panel 已保存: {out_panel_path}")

    if save_removed_detail and removed_detail_path is not None:
        removed_detail.to_csv(removed_detail_path, index=False, encoding="utf-8-sig")
        print(f"[INFO] 被删除样本明细已保存: {removed_detail_path}")

    if removed_count > 0:
        show_cols = [c for c in [date_col, stock_col, "ref_date", "ref_value"] if c in removed_detail.columns]
        print("\n[被删除样本前10行示例]")
        print(removed_detail[show_cols].head(10).to_string(index=False))
    else:
        print("[INFO] 当前没有样本被删除，请检查代码对齐口径。")

    return filtered_panel, removed_detail


def main():
    start_time = datetime.now()
    print(f"开始时间: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # 1) 读取参考表并转成长表
    ref_long = load_reference_table(REF_CSV_PATH)

    # 2) 读取 panel
    panel_df, stock_col, date_col = load_panel(PANEL_PATH)

    print(f"[INFO] panel 股票列: {stock_col}")
    print(f"[INFO] panel 日期列: {date_col}")
    print(f"[INFO] panel 总行数: {len(panel_df):,}")
    print(f"[INFO] panel 唯一股票数: {panel_df[stock_col].nunique():,}")
    print(f"[INFO] panel 日期数: {panel_df[date_col].nunique():,}")

    # 3) 给 panel 每一行匹配“上一个有效交易日”的 14:30 涨幅
    merged_df = attach_previous_valid_value(
        panel_df=panel_df,
        stock_col=stock_col,
        date_col=date_col,
        ref_long=ref_long
    )

    # 4) 只筛 panel，不再同步筛选文本 / NPY
    filter_panel(
        panel_with_ref=merged_df,
        stock_col=stock_col,
        date_col=date_col,
        out_panel_path=OUT_PANEL_PATH,
        removed_detail_path=REMOVED_DETAIL_PATH,
        save_removed_detail=SAVE_REMOVED_DETAIL
    )

    end_time = datetime.now()
    seconds = (end_time - start_time).total_seconds()
    print(f"\n总耗时: {seconds:.2f} 秒")


if __name__ == "__main__":
    main()

开始时间: 2026-04-03 16:39:04
[INFO] 正在读取参考表: D:\PythonProject\LASSO_FINAL\data\14.30相对前一日收盘涨幅.csv
[INFO] 参考表共有 3185 只股票列，开始转成长表...
[INFO] 长表转换完成，有效记录数: 2,907,162
[INFO] 正在读取 panel: D:\PythonProject\LASSO_FINAL\data\panel_raw_selected_noST.pkl
[INFO] panel 股票列: ts_code
[INFO] panel 日期列: date
[INFO] panel 总行数: 2,789,071
[INFO] panel 唯一股票数: 3,058
[INFO] panel 日期数: 936
[INFO] 正在匹配每一行对应的“上一个有效交易日”的 14:30 涨幅...
[INFO] 匹配完成：
       成功匹配到上一个有效交易日的行数: 2786013
       因没有更早有效交易日而未匹配的行数: 3058
       因参考表中不存在该股票而未匹配的行数: 0

================ 筛选统计 ================
原始样本数: 2,789,071
删除样本数: 44,969
保留样本数: 2,744,102

[INFO] 筛选后的 panel 已保存: D:\PythonProject\LASSO_FINAL\data\panel_raw_selected_noST_no9pct.pkl
[INFO] 被删除样本明细已保存: D:\PythonProject\LASSO_FINAL\data\panel_removed_by_1430_limit_detail.csv

[被删除样本前10行示例]
      date   ts_code   ref_date  ref_value
2022-11-30 000001.SZ 2022-11-29   0.093141
2024-02-22 000001.SZ 2024-02-21   0.099796
2022-11-14 000002.SZ 2022-11-11   0.090021
2022-11-30 000002.SZ 2022-11